In [0]:
spark.sql("USE CATALOG 03_gold")
spark.sql("USE SCHEMA default")
from pyspark.sql.functions import (
    col, initcap, regexp_replace, to_date, coalesce, 
    current_timestamp, expr, trim
)
import re

In [0]:
df = spark.read.table("02_silver.default.customers")

df_dim_customer = df.select(
    col("customer_id"),
    col("customer_name"),
    col("phone_number"),
    col("gender")
).dropDuplicates(["customer_id"])

spark.sql("DROP TABLE IF EXISTS dim_customer")

df_dim_customer.write.format("delta").mode("overwrite").saveAsTable("dim_customer")

In [0]:
df = spark.read.table("02_silver.default.products")

df_dim_product = df.select(
    col("product_id"),
    col("product_name"),
    col("category"),
    col("cost_price")
).dropDuplicates(["product_id"])

spark.sql("DROP TABLE IF EXISTS dim_product")

df_dim_product.write.format("delta").mode("overwrite").saveAsTable("dim_product")

In [0]:
df = spark.read.table("02_silver.default.stores")

df_dim_store = df.select(
    col("store_id"),
    col("store_name"),
    col("city")
).dropDuplicates(["store_id"])

spark.sql("DROP TABLE IF EXISTS dim_store")

df_dim_store.write.format("delta").mode("overwrite").saveAsTable("dim_store")

In [0]:
spark.sql("DROP TABLE IF EXISTS 03_gold.default.fact_sales")
df_sales = spark.read.table("02_silver.default.sales")

df_fact = df_sales.select(
    col("transaction_id"),
    col("customer_id"),
    col("store_id"),
    col("product_id"),   # ✅ now exists
    col("transaction_date"),
    col("quantity"),
    col("selling_price"),
    col("cost_price"),
    col("discount"),
    
    (col("quantity") * col("selling_price")).alias("total_sales"),
    (col("selling_price") - col("cost_price")).alias("profit_per_unit"),
    (col("quantity") * (col("selling_price") - col("cost_price"))).alias("total_profit")
)

df_fact.write.format("delta").mode("overwrite").saveAsTable("03_gold.default.fact_sales")